# Telegram Media Downloader v2

A powerful CLI tool that bulk-downloads all media (photos, videos, documents, audio, voice, video notes, animations, stickers) from your Telegram channels and groups — **including restricted/protected content** from private channels.

Built with **PyroFork** (Pyrogram fork) using your own Telegram user account via MTProto API.

**Platforms:** Windows · macOS · Linux · Google Colab

## 1. Standard Library Imports

In [ ]:
import asyncio
import io
import json
import os
import re
import subprocess
import sys
import time
import logging
from datetime import datetime

## 2. Environment Detection & Setup

Detects Google Colab vs local machine:
- **Colab**: Auto-installs dependencies (`pyrofork`, `TgCrypto-pyrofork`, `nest_asyncio`, `uvloop`), patches the running event loop, enables `uvloop` for faster async I/O, sets download directory to `/content/telegram_downloader/`
- **Windows**: Wraps stdout/stderr with UTF-8 encoding
- **Local**: Uses script directory as base

In [ ]:
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ

if IS_COLAB:
    # Auto-install dependencies
    print("  [COLAB] Detected Google Colab environment")
    print("  [COLAB] Installing dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pyrofork", "TgCrypto-pyrofork", "nest_asyncio", "uvloop"])

    # Patch asyncio for Colab (Colab already has a running event loop)
    import nest_asyncio
    nest_asyncio.apply()

    # Use uvloop for 2-4x faster async I/O (Linux only)
    try:
        import uvloop
        uvloop.install()
        print("  [COLAB] uvloop installed (faster async I/O)")
    except Exception:
        pass

    # Download to Colab session storage (fast local disk)
    BASE_DIR = "/content/telegram_downloader"
    os.makedirs(BASE_DIR, exist_ok=True)
    print(f"  [COLAB] Working directory: {BASE_DIR}")
    print(f"  [COLAB] Downloads saved to Colab session disk")
    print()
else:
    # Local machine
    if sys.platform == "win32":
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")
        sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding="utf-8", errors="replace")

    BASE_DIR = os.path.dirname(os.path.abspath(__file__))

## 3. Pyrogram Imports & Logging

Imports PyroFork client, enums, message types, and error classes.

`FloodPremiumWait` is imported with a fallback to `FloodWait` for older PyroFork versions.
Pyrogram logs are set to WARNING (to see FloodWait retries); asyncio logs are suppressed.

In [ ]:
from pyrogram import Client
from pyrogram.enums import ChatType, MessageMediaType, MessagesFilter
from pyrogram.types import InputMediaDocument, InputMediaPhoto, InputMediaVideo, InputMediaAudio
from pyrogram.errors import (
    ChannelPrivate,
    ChatAdminRequired,
    FileReferenceExpired,
    FloodWait,
    RPCError,
)

# FloodPremiumWait is raised for free accounts during large uploads
try:
    from pyrogram.errors import FloodPremiumWait
except ImportError:
    FloodPremiumWait = FloodWait  # fallback if not available

# Show Pyrogram warnings during uploads (FloodPremiumWait retries visible at WARNING level)
# Change to CRITICAL to suppress all Pyrogram logs
logging.getLogger("pyrogram").setLevel(logging.WARNING)
# Suppress asyncio "socket.send() raised exception" and "Task exception was never retrieved" warnings
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

## 4. Compatibility Patches

### 4.1 SQLite Timeout Patch (Colab)

Pyrogram uses `sqlite3.connect(timeout=1)` — only 1 second. On Colab re-runs (same kernel), the old client still holds the SQLite lock. This patch:

1. Increases default timeout to **30 seconds**
2. Patches both the `sqlite3` module and Pyrogram's storage module
3. Uses a `_tgdl_patched` flag to prevent double-patching (which would cause `RecursionError`)

In [ ]:
# Pyrogram uses sqlite3.connect(path, timeout=1) — only 1 second.
# On Colab re-runs, the old client is still alive in the same event loop,
# holding the lock. Patch to 30s so operations wait instead of crashing.
# Must patch BOTH the sqlite3 module AND Pyrogram's storage module directly,
# since Pyrogram already imported sqlite3 at this point.
import sqlite3 as _sqlite3

# Guard against double-patching on Colab re-runs (same kernel)
if not getattr(_sqlite3, '_tgdl_patched', False):
    _original_sqlite3_connect = _sqlite3.connect

    def _sqlite3_connect_with_timeout(*args, **kwargs):
        kwargs.setdefault("timeout", 30)
        return _original_sqlite3_connect(*args, **kwargs)

    _sqlite3.connect = _sqlite3_connect_with_timeout
    _sqlite3._tgdl_patched = True

    # Also patch Pyrogram's own reference to sqlite3 in its storage module
    try:
        import pyrogram.storage.sqlite_storage as _ss_mod
        _ss_mod.sqlite3.connect = _sqlite3_connect_with_timeout
    except Exception:
        pass

# Track the active client globally so we can stop it on Colab re-runs
_active_client = None

### 4.2 Upload Pipeline Patch (Queue(1) → Queue(16))

Pyrogram's `save_file()` uses `asyncio.Queue(1)`, meaning only **one** 512KB chunk is in-flight at a time. This serializes uploads to: send → wait ACK → send next.

This patch widens the queue to `Queue(16)`, allowing 16 chunks in-flight through 4 workers, pipelining the upload for **3-8x speed improvement**.

> Only patched in the `save_file` module — does NOT affect any other `asyncio.Queue` usage.

In [ ]:
# Pyrogram's save_file() uses asyncio.Queue(1), meaning only ONE 512KB chunk
# is in-flight at a time. This serializes uploads to: send → wait ACK → send next.
# Patching to Queue(16) allows 16 chunks in-flight simultaneously through the
# 4 upload workers, pipelining the upload for 3-8x speed improvement.
# ONLY patched in save_file module — does NOT affect any other asyncio.Queue usage.
try:
    import types as _types
    import pyrogram.methods.advanced.save_file as _sf_mod

    _OrigQueue = asyncio.Queue

    class _PipelinedQueue(_OrigQueue):
        """Upload queue widened from 1→16 for pipelined chunk uploads."""
        def __init__(self, maxsize=0):
            super().__init__(16 if maxsize == 1 else maxsize)

    # Create a proxy asyncio module with only Queue replaced
    _proxy_asyncio = _types.ModuleType("asyncio")
    _proxy_asyncio.__dict__.update(asyncio.__dict__)
    _proxy_asyncio.Queue = _PipelinedQueue
    _sf_mod.asyncio = _proxy_asyncio
except Exception:
    pass

## 5. Constants & Configuration

### 5.1 Paths, Thresholds & Delays

| Constant Group | Description |
|---|---|
| `DOWNLOAD_DIR`, `TRACKER_FILE`, etc. | File paths for downloads, trackers, config, session |
| `CHUNK_SIZE`, `LARGE_FILE_THRESHOLD` | Chunked download settings (1 MB chunks, 50 MB threshold) |
| `INTER_DELAY_*` | Adaptive inter-file download delays (0.5s–10s) to reduce FloodWait |
| `SEND_DELAY_*` | Upload cooldown delays by file size |
| `UPLOAD_LIMIT_*` | Telegram upload limits (2 GB free, 4 GB premium) |

In [ ]:
DOWNLOAD_DIR = os.path.join(BASE_DIR, "telegram_downloads")
TRACKER_FILE = os.path.join(BASE_DIR, "download_tracker.json")
SENT_TRACKER_FILE = os.path.join(BASE_DIR, "sent_tracker.json")
CONFIG_FILE = os.path.join(BASE_DIR, "tg_config.json")
SESSION_NAME = os.path.join(BASE_DIR, "my_telegram_session")

SAVE_EVERY = 10
CHUNK_SIZE = 1024 * 1024  # 1 MB per chunk
LARGE_FILE_THRESHOLD = 50 * 1024 * 1024  # 50 MB
ESTIMATED_SPEED = (5 * 1024 * 1024) if IS_COLAB else (200 * 1024)  # 5 MB/s Colab, 200 KB/s local
SEND_DELAY_SMALL = 3   # seconds delay for files < 100 MB
SEND_DELAY_MEDIUM = 10  # seconds delay for files 100 MB - 500 MB
SEND_DELAY_LARGE = 20   # seconds delay for files > 500 MB
UPLOAD_LIMIT_FREE = 2 * 1024 * 1024 * 1024      # 2 GB
UPLOAD_LIMIT_PREMIUM = 4 * 1024 * 1024 * 1024    # 4 GB

# Upload tuning
UPLOAD_FLOOD_RETRY_CAP = 10   # max FloodWait retries per file before skipping
UPLOAD_BATCH_SIZE = 10        # max files per send_media_group batch
UPLOAD_CONCURRENT = 3         # max parallel file uploads
UPLOAD_SAVE_EVERY = 10        # save sent_tracker every N successful uploads
PHOTO_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm"}
AUDIO_EXTENSIONS = {".mp3", ".ogg", ".flac", ".wav", ".m4a"}

# Adaptive inter-file download delays (seconds) — reduces FloodWait risk
INTER_DELAY_TINY = 0.5    # < 1 MB (photos)
INTER_DELAY_SMALL = 2     # 1-10 MB
INTER_DELAY_MEDIUM = 3    # 10-50 MB
INTER_DELAY_LARGE = 5     # 50-500 MB
INTER_DELAY_XLARGE = 10   # > 500 MB

### 5.2 File Naming, Session State & Media Mappings

- `FILE_NAME_TEMPLATE` — Custom naming with `{filename}`, `{date}`, `{msgid}`, `{type}`, `{ext}` placeholders
- `session_failed_files` — Session-level list of failed downloads for retry
- `MEDIA_EXTENSIONS` — Default extensions per media type
- `MEDIA_FILTER_MAP` — Maps human-readable type names to Pyrogram `MessagesFilter` enums

In [ ]:
# File naming template: {filename}, {ext}, {msgid}, {date}, {date_time}, {type}
# Default: original filename. Change to e.g. "{date}_{msgid}_{filename}" for sorted names
FILE_NAME_TEMPLATE = ""  # empty = use original filename as-is

# Session-level state
session_failed_files = []
user_is_premium = False

# Media types that are typically large and need individual file selection
LARGE_MEDIA_TYPES = {"videos", "documents"}

MEDIA_EXTENSIONS = {
    MessageMediaType.PHOTO: ".jpg",
    MessageMediaType.VIDEO: ".mp4",
    MessageMediaType.AUDIO: ".mp3",
    MessageMediaType.VOICE: ".ogg",
    MessageMediaType.VIDEO_NOTE: ".mp4",
    MessageMediaType.ANIMATION: ".mp4",
    MessageMediaType.STICKER: ".webp",
}

# Map media types to MessagesFilter for search_messages / search_messages_count
MEDIA_FILTER_MAP = {
    "photos": MessagesFilter.PHOTO,
    "videos": MessagesFilter.VIDEO,
    "documents": MessagesFilter.DOCUMENT,
    "audio": MessagesFilter.AUDIO,
    "voice": MessagesFilter.VOICE_NOTE,
    "video_notes": MessagesFilter.VIDEO_NOTE,
    "gifs": MessagesFilter.ANIMATION,
}

## 6. Utility Functions

### 6.1 Filename Sanitization

`sanitize_name()` — Strips characters invalid on Windows/macOS/Linux (`<>:"/\|?*`).
Handles Windows reserved names (CON, PRN, AUX, NUL, COM1-9, LPT1-9) by prefixing with `_`.

In [ ]:
_WINDOWS_RESERVED = frozenset({
    "CON", "PRN", "AUX", "NUL",
    *(f"COM{i}" for i in range(1, 10)),
    *(f"LPT{i}" for i in range(1, 10)),
})


def sanitize_name(name):
    """Remove characters invalid on Windows/macOS/Linux from file/dir names."""
    if not name:
        return "Unnamed"
    name = re.sub(r'[<>:"/\\|?*]', "", name)
    name = name.strip(". ")
    if not name:
        return "Unnamed"
    # Windows reserved names: CON, PRN, AUX, NUL, COM1-COM9, LPT1-LPT9
    stem = name.split(".")[0].upper()
    if stem in _WINDOWS_RESERVED:
        name = f"_{name}"
    return name

### 6.2 File Name & Size Extraction

- `get_file_name()` — Extracts filename from message media attributes (document, audio, video, animation, sticker), with fallback to `{type}_{msgid}.{ext}`
- `get_file_size()` — Gets file size from the appropriate media attribute (photo, document, video, etc.)

In [ ]:
def get_file_name(message):
    """Extract or generate a filename for the media in a message."""
    media = message.media
    msg_id = message.id

    if message.document and message.document.file_name:
        return message.document.file_name
    if message.audio and message.audio.file_name:
        return message.audio.file_name
    if message.video and message.video.file_name:
        return message.video.file_name
    if message.animation and message.animation.file_name:
        return message.animation.file_name

    if media == MessageMediaType.STICKER and message.sticker:
        if message.sticker.is_animated:
            return f"sticker_{msg_id}.tgs"
        elif message.sticker.is_video:
            return f"sticker_{msg_id}.webm"
        else:
            return f"sticker_{msg_id}.webp"

    ext = MEDIA_EXTENSIONS.get(media, ".bin")
    type_name = media.name.lower() if media else "file"
    return f"{type_name}_{msg_id}{ext}"


def get_file_size(message):
    """Get the file size from message media attributes."""
    if message.photo:
        return getattr(message.photo, "file_size", 0) or 0
    for attr in ("document", "video", "audio", "voice", "video_note", "animation", "sticker"):
        media_obj = getattr(message, attr, None)
        if media_obj and hasattr(media_obj, "file_size") and media_obj.file_size:
            return media_obj.file_size
    return 0

### 6.3 Formatting & Progress Display

- `format_size()` — Bytes to human-readable (B → KB → MB → GB → TB → PB)
- `format_time()` — Seconds to `Xh Xm Xs` format
- `print_progress()` — Real-time inline progress bar with percentage, size, and speed

In [ ]:
def format_size(size_bytes):
    """Convert bytes to human-readable format."""
    if size_bytes == 0:
        return "0 B"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(size_bytes) < 1024:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024
    return f"{size_bytes:.2f} PB"


def format_time(seconds):
    """Convert seconds to human-readable duration."""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        m, s = divmod(int(seconds), 60)
        return f"{m}m {s}s"
    else:
        h, remainder = divmod(int(seconds), 3600)
        m, s = divmod(remainder, 60)
        return f"{h}h {m}m {s}s"


def print_progress(file_name, current, total, start_time):
    """Display real-time download progress."""
    now = time.time()
    elapsed = now - start_time
    speed = current / elapsed if elapsed > 0 else 0
    percent = (current / total * 100) if total > 0 else 0
    display_name = file_name if len(file_name) <= 35 else file_name[:32] + "..."

    print(
        f"\r  >> {display_name} | {percent:5.1f}% | "
        f"{format_size(current)}/{format_size(total)} | "
        f"{format_size(speed)}/s   ",
        end="",
        flush=True,
    )
    if current >= total:
        print()

### 6.4 File Naming Template

`apply_name_template()` — Applies `FILE_NAME_TEMPLATE` with placeholders:
`{filename}`, `{ext}`, `{msgid}`, `{date}` (YYYY-MM-DD), `{date_time}` (YYYY-MM-DD_HHMMSS), `{type}`

In [ ]:
def apply_name_template(template, message, original_name):
    """Apply naming template to generate filename. Returns original_name if template is empty."""
    if not template:
        return original_name
    name_part, ext = os.path.splitext(original_name)
    media_type = message.media.name.lower() if message.media else "file"
    msg_date = message.date
    date_str = msg_date.strftime("%Y-%m-%d") if msg_date else "unknown"
    datetime_str = msg_date.strftime("%Y-%m-%d_%H%M%S") if msg_date else "unknown"
    try:
        result = template.format(
            filename=name_part,
            ext=ext,
            msgid=message.id,
            date=date_str,
            date_time=datetime_str,
            type=media_type,
        )
        # Ensure extension is present
        if not result.endswith(ext):
            result += ext
        return sanitize_name(result)
    except (KeyError, ValueError):
        return original_name

### 6.5 User Selection Parser

`parse_selection()` — Parses user input like `1,3,5` or `1-10` or `all` into sorted 0-based indices.
Returns `None` on invalid input.

In [ ]:
def parse_selection(selection_str, max_val):
    """Parse user selection like '1,3,5' or '1-10' or 'all'. Returns sorted list of 0-based indices."""
    selection_str = selection_str.strip().lower()
    if not selection_str:
        return None
    if selection_str == "all":
        return list(range(max_val))

    indices = set()
    try:
        for part in selection_str.split(","):
            part = part.strip()
            if "-" in part:
                start, end = part.split("-", 1)
                start, end = int(start.strip()), int(end.strip())
                if start < 1 or end > max_val or start > end:
                    return None
                indices.update(range(start - 1, end))
            else:
                num = int(part)
                if num < 1 or num > max_val:
                    return None
                indices.add(num - 1)
    except ValueError:
        return None

    return sorted(indices) if indices else None

## 7. Download Functions

### 7.1 Chunked Download (Large Files ≥ 50 MB)

`chunked_download()` — Uses `stream_media(offset=N)` with 1 MB chunks.

**Resume support:** Interrupted downloads leave `.part` files. On resume, the file is truncated to the nearest chunk boundary and downloading continues from there.

**Error handling:**
- `FileReferenceExpired` — Re-fetches message, resumes from current position (3 retries)
- `FloodWait` / `FloodPremiumWait` — Sleeps and retries (30 retries for large files)

In [ ]:
async def chunked_download(client, message, file_path, file_name, chat_id):
    """Download large files using stream_media() with chunk-based resume."""
    total_size = get_file_size(message)
    temp_path = file_path + ".part"

    existing_size = 0
    start_chunk = 0
    if os.path.exists(temp_path):
        existing_size = os.path.getsize(temp_path)
        start_chunk = existing_size // CHUNK_SIZE
        # Truncate to chunk boundary to avoid duplicate bytes from incomplete chunks
        aligned_size = start_chunk * CHUNK_SIZE
        if existing_size != aligned_size:
            with open(temp_path, "r+b") as tf:
                tf.truncate(aligned_size)
            existing_size = aligned_size
        if existing_size > 0:
            print(f"  -- Resuming from {format_size(existing_size)}...")

    start_time = time.time()
    current_msg = message
    written = existing_size
    last_progress = 0

    max_ref_retries = 3   # FileReferenceExpired retries (limited)
    max_flood_retries = 30  # FloodWait/FloodPremiumWait retries (generous — large files get throttled many times)
    ref_retries = 0
    flood_retries = 0

    while True:
        try:
            mode = "ab" if start_chunk > 0 else "wb"
            with open(temp_path, mode) as f:
                async for chunk in client.stream_media(current_msg, offset=start_chunk):
                    f.write(chunk)
                    written += len(chunk)

                    now = time.time()
                    if now - last_progress >= 0.3 or written >= total_size:
                        display_total = total_size if total_size > 0 else written
                        print_progress(file_name, written, display_total, start_time)
                        last_progress = now

            if os.path.exists(file_path):
                os.remove(file_path)
            os.rename(temp_path, file_path)
            return file_path

        except FileReferenceExpired:
            ref_retries += 1
            if ref_retries > max_ref_retries:
                raise Exception(f"Download failed after {max_ref_retries} file reference refreshes")
            print(f"\n  !! File reference expired -- refreshing ({ref_retries}/{max_ref_retries})...")
            try:
                current_msg = await client.get_messages(chat_id, message.id)
                if os.path.exists(temp_path):
                    existing_size = os.path.getsize(temp_path)
                    start_chunk = existing_size // CHUNK_SIZE
                    written = existing_size
            except Exception as e:
                print(f"  !! Failed to refresh message: {e}")
                raise

        except (FloodPremiumWait, FloodWait) as e:
            flood_retries += 1
            wait_time = getattr(e, "value", 10)
            error_name = type(e).__name__
            if flood_retries > max_flood_retries:
                raise Exception(f"Download failed after {max_flood_retries} flood waits")
            pct = (written / total_size * 100) if total_size > 0 else 0
            print(f"\n  !! {error_name}: waiting {wait_time}s (throttle #{flood_retries}, {pct:.1f}% done)...")
            await asyncio.sleep(wait_time)
            if os.path.exists(temp_path):
                existing_size = os.path.getsize(temp_path)
                start_chunk = existing_size // CHUNK_SIZE
                written = existing_size

### 7.2 Standard Download (Small Files < 50 MB)

`standard_download()` — Uses Pyrogram's built-in `download_media()` with progress callback.

**Error handling:**
- `FileReferenceExpired` — Re-fetches message and retries (3 attempts)
- `FloodWait` / `FloodPremiumWait` — Sleeps and retries (15 attempts)

In [ ]:
async def standard_download(client, message, file_path, file_name, chat_id):
    """Download smaller files using download_media() with FileReferenceExpired handling."""
    start_time = time.time()
    ref_retries = 0
    flood_retries = 0
    need_refresh = False

    while True:
        try:
            current_msg = message if (ref_retries == 0 and not need_refresh) else await client.get_messages(chat_id, message.id)
            need_refresh = False
            path = await client.download_media(
                current_msg,
                file_name=file_path,
                progress=lambda current, total: print_progress(
                    file_name, current, total, start_time
                ),
            )
            return path

        except FileReferenceExpired:
            ref_retries += 1
            if ref_retries > 3:
                return None
            print(f"\n  !! File reference expired -- refreshing ({ref_retries}/3)...")
            need_refresh = True
            await asyncio.sleep(1)

        except (FloodPremiumWait, FloodWait) as e:
            flood_retries += 1
            wait_time = getattr(e, "value", 10)
            error_name = type(e).__name__
            if flood_retries > 15:
                print(f"\n  !! Too many flood waits ({flood_retries}). Skipping file.")
                return None
            print(f"\n  !! {error_name}: waiting {wait_time}s (throttle #{flood_retries})...")
            need_refresh = True
            await asyncio.sleep(wait_time)

## 8. Configuration & Authentication

### 8.1 Config File Management

- `load_config()` — Loads `api_id` and `api_hash` from `tg_config.json`
- `save_config()` — Saves credentials to `tg_config.json`

In [ ]:
def load_config():
    """Load saved API credentials from config file."""
    if os.path.exists(CONFIG_FILE):
        try:
            with open(CONFIG_FILE, "r") as f:
                config = json.load(f)
            if config.get("api_id") and config.get("api_hash"):
                return config["api_id"], config["api_hash"]
        except (json.JSONDecodeError, KeyError):
            pass
    return None, None


def save_config(api_id, api_hash):
    """Save API credentials to config file."""
    with open(CONFIG_FILE, "w") as f:
        json.dump({"api_id": api_id, "api_hash": api_hash}, f, indent=2)
    print(f"  [OK] Credentials saved to {CONFIG_FILE}")

### 8.2 Client Setup & Authentication

`setup_client()` — Creates and authenticates the Pyrogram client:

1. Loads or prompts for API credentials
2. On Colab re-runs: stops old client, clears stale SQLite locks (WAL/SHM/journal files)
3. Creates `Client()` with `max_concurrent_transmissions=3`, `sleep_threshold=120`
4. Handles "database is locked" errors with retry
5. Detects Premium vs Free account (affects upload limits)

In [ ]:
async def setup_client():
    """Create and authenticate the Pyrogram client."""
    api_id, api_hash = load_config()

    if api_id and api_hash:
        print(f"  [OK] Loaded credentials from {CONFIG_FILE}")
    else:
        print("\n  First-time setup -- enter your Telegram API credentials.")
        print("  (Get them from https://my.telegram.org)\n")

        while True:
            try:
                api_id = int(input("  Enter API ID: ").strip())
                break
            except ValueError:
                print("  [ERROR] API ID must be a number. Try again.")

        api_hash = input("  Enter API Hash: ").strip()
        if not api_hash:
            print("  [ERROR] API Hash cannot be empty.")
            sys.exit(1)

        save_choice = input("\n  Save credentials for future runs? (y/n): ").strip().lower()
        if save_choice == "y":
            save_config(api_id, api_hash)

    # On Colab re-runs, stop the old client first to release the SQLite lock
    global _active_client
    if _active_client is not None:
        print("  [INFO] Stopping previous client session...")
        try:
            await _active_client.stop()
        except Exception:
            pass
        _active_client = None
        await asyncio.sleep(1)

    # Force-close any stale SQLite locks from crashed sessions (Colab re-runs)
    session_file = SESSION_NAME + ".session"
    if os.path.exists(session_file):
        for suffix in ["-journal", "-wal", "-shm"]:
            lock_file = session_file + suffix
            if os.path.exists(lock_file):
                try:
                    os.remove(lock_file)
                except Exception:
                    pass
        # Open and immediately close to force SQLite to release any stale POSIX lock
        try:
            _conn = _sqlite3.connect(session_file, timeout=1)
            _conn.execute("PRAGMA wal_checkpoint(TRUNCATE)")
            _conn.close()
        except Exception:
            pass

    client = Client(
        name=SESSION_NAME,
        api_id=api_id,
        api_hash=api_hash,
        max_concurrent_transmissions=3,
        sleep_threshold=120,  # Let Pyrogram handle FloodWait ≤120s internally (save_file workers retry instead of skipping chunks)
    )

    print("\n  Connecting to Telegram...")
    try:
        await client.start()
    except Exception as e:
        if "database is locked" in str(e):
            print("  [WARN] Session locked by previous run. Waiting for lock release...")
            await asyncio.sleep(5)
            await client.start()
        else:
            raise
    _active_client = client

    me = await client.get_me()
    name = me.first_name or ""
    if me.last_name:
        name += f" {me.last_name}"

    global user_is_premium
    user_is_premium = getattr(me, "is_premium", False) or False
    acct_type = "Premium" if user_is_premium else "Free"
    print(f"  [OK] Logged in as: {name} (@{me.username or 'N/A'}) [{acct_type}]\n")

    return client

## 9. Download Tracker

Tracks downloaded message IDs per channel in `download_tracker.json`.

- `load_tracker()` / `save_tracker()` — Disk I/O with corruption recovery
- `is_downloaded()` — O(1) lookup via cached `_ids_set` (lazy-initialized)
- `mark_downloaded()` — Appends ID, updates set cache, periodic saves every 10 downloads

In [ ]:
def load_tracker():
    """Load the download tracker from disk."""
    if os.path.exists(TRACKER_FILE):
        try:
            with open(TRACKER_FILE, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, ValueError):
            print("  [WARN] Tracker file corrupted -- starting fresh.")
    return {}


def save_tracker(tracker):
    """Save the download tracker to disk."""
    # Strip cached sets before serializing
    clean = {}
    for k, v in tracker.items():
        clean[k] = {ik: iv for ik, iv in v.items() if ik != "_ids_set"}
    with open(TRACKER_FILE, "w") as f:
        json.dump(clean, f, indent=2)


def is_downloaded(tracker, chat_id, message_id):
    """Check if a message has already been downloaded."""
    chat_key = str(chat_id)
    if chat_key in tracker:
        ids = tracker[chat_key].get("downloaded_ids", [])
        # Use set for O(1) lookup if not already cached
        if "_ids_set" not in tracker[chat_key]:
            tracker[chat_key]["_ids_set"] = set(ids)
        return message_id in tracker[chat_key]["_ids_set"]
    return False


def mark_downloaded(tracker, chat_id, chat_title, message_id, download_count):
    """Mark a message as downloaded and periodically save."""
    chat_key = str(chat_id)
    if chat_key not in tracker:
        tracker[chat_key] = {
            "title": chat_title,
            "downloaded_ids": [],
            "last_updated": None,
        }
    tracker[chat_key]["downloaded_ids"].append(message_id)
    if "_ids_set" in tracker[chat_key]:
        tracker[chat_key]["_ids_set"].add(message_id)
    tracker[chat_key]["last_updated"] = datetime.now().isoformat()

    if download_count % SAVE_EVERY == 0:
        save_tracker(tracker)

## 10. Sent-to-Saved Tracker

### 10.1 Tracker CRUD Operations

Tracks files sent to Saved Messages in `sent_tracker.json`.
Same O(1) cached `_files_set` pattern as the download tracker.

- `load_sent_tracker()` / `save_sent_tracker()` — Disk I/O (strips cache before serializing)
- `is_already_sent()` — O(1) lookup via cached set
- `mark_as_sent()` — Appends filename, updates cache

In [ ]:
def load_sent_tracker():
    """Load the sent-to-saved tracker from disk."""
    if os.path.exists(SENT_TRACKER_FILE):
        try:
            with open(SENT_TRACKER_FILE, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, ValueError):
            pass
    return {}


def save_sent_tracker(sent_tracker):
    """Save the sent-to-saved tracker to disk."""
    clean = {}
    for k, v in sent_tracker.items():
        clean[k] = {ik: iv for ik, iv in v.items() if ik != "_files_set"}
    with open(SENT_TRACKER_FILE, "w") as f:
        json.dump(clean, f, indent=2)


def is_already_sent(sent_tracker, channel_name, file_name):
    """Check if a file was already sent to Saved Messages (O(1) via cached set)."""
    if channel_name not in sent_tracker:
        return False
    ch = sent_tracker[channel_name]
    if "_files_set" not in ch:
        ch["_files_set"] = set(ch.get("files", []))
    return file_name in ch["_files_set"]


def mark_as_sent(sent_tracker, channel_name, file_name):
    """Mark a file as sent to Saved Messages."""
    if channel_name not in sent_tracker:
        sent_tracker[channel_name] = {"files": [], "last_updated": None}
    sent_tracker[channel_name]["files"].append(file_name)
    if "_files_set" in sent_tracker[channel_name]:
        sent_tracker[channel_name]["_files_set"].add(file_name)
    sent_tracker[channel_name]["last_updated"] = datetime.now().isoformat()

### 10.2 Saved Messages Scanning & Recovery

Recovers sent-tracker state by scanning Saved Messages (e.g., after Colab session loss).

- `scan_saved_for_channel()` — Searches Saved Messages for files from a specific channel using caption format `ChannelName/filename`
- `rebuild_sent_tracker_for_channels()` — Scans multiple channels and merges newly discovered files into the tracker

In [ ]:
async def scan_saved_for_channel(client, channel_name):
    """Scan Saved Messages to find files already uploaded from a channel.

    Uses the caption format 'ChannelName/filename' (set during upload) to identify
    which files from this channel are already in Saved Messages.
    This recovers state even if sent_tracker.json was lost (e.g., Colab session crash).

    Returns a set of filenames found in Saved Messages for this channel.
    """
    found_files = set()
    prefix = f"{channel_name}/"

    for attempt in range(2):
        try:
            async for msg in client.search_messages("me", query=channel_name):
                if msg.caption and msg.caption.startswith(prefix):
                    fname = msg.caption[len(prefix):]
                    if fname:
                        found_files.add(fname)
            break
        except (FloodWait, FloodPremiumWait) as e:
            wait_time = getattr(e, "value", 10)
            if attempt == 0:
                print(f"    FloodWait: sleeping {wait_time}s then retrying scan...")
                await asyncio.sleep(wait_time)
            else:
                break
        except Exception as e:
            print(f"    [WARN] Could not scan Saved Messages for {channel_name}: {e}")
            break

    return found_files


async def rebuild_sent_tracker_for_channels(client, sent_tracker, channel_names):
    """Scan Saved Messages and merge found files into sent_tracker.

    For each channel, searches Saved Messages for files uploaded by this tool
    (identified by caption format). Merges any newly discovered files into the
    tracker so they'll be skipped during upload.

    Returns dict: {channel_name: number_of_new_files_found}
    """
    results = {}
    print("\n  Scanning Saved Messages to detect already-uploaded files...")

    for ch_name in channel_names:
        print(f"    Scanning: {ch_name}...", end=" ", flush=True)
        found = await scan_saved_for_channel(client, ch_name)

        if found:
            if ch_name not in sent_tracker:
                sent_tracker[ch_name] = {"files": [], "last_updated": None}
            existing = set(sent_tracker[ch_name]["files"])
            new_found = found - existing

            if new_found:
                sent_tracker[ch_name]["files"].extend(sorted(new_found))
                sent_tracker[ch_name]["last_updated"] = datetime.now().isoformat()
                print(f"found {len(found)} ({len(new_found)} recovered)")
            else:
                print(f"found {len(found)} (all already tracked)")
            results[ch_name] = len(new_found)
        else:
            print(f"none found")
            results[ch_name] = 0

    total_recovered = sum(results.values())
    if total_recovered > 0:
        save_sent_tracker(sent_tracker)
        print(f"  Recovered {total_recovered} file(s) into sent tracker.\n")
    else:
        print(f"  Tracker is up to date.\n")

    return results

## 11. Chat Discovery & Display

- `list_chats()` — Fetches all channels/groups via `get_dialogs()`, excludes private chats and bots. Captures chat ID, title, type, member count, and protection status.
- `display_chats()` — Formatted table with type, name, member count, and `[LOCKED]` tag for protected channels.

In [ ]:
async def list_chats(client):
    """Fetch all channels and groups the user is a member of."""
    chats = []
    print("  Fetching your channels and groups...\n")

    async for dialog in client.get_dialogs():
        chat = dialog.chat

        if chat.type in (ChatType.PRIVATE, ChatType.BOT):
            continue

        chat_type = {
            ChatType.GROUP: "Group",
            ChatType.SUPERGROUP: "Supergroup",
            ChatType.CHANNEL: "Channel",
        }.get(chat.type, "Unknown")

        chats.append({
            "id": chat.id,
            "title": chat.title or "Unnamed",
            "type": chat_type,
            "members": getattr(chat, "members_count", None) or "?",
            "restricted": getattr(chat, "has_protected_content", False),
        })

    return chats


def display_chats(chats):
    """Display chats in a formatted table."""
    if not chats:
        print("  [ERROR] No channels or groups found.")
        return

    print(f"  Found {len(chats)} channels/groups:\n")
    print(f"  {'#':>4}  {'Type':<12} {'Name':<40} {'Members':>8}  {'Protected'}")
    print(f"  {'-'*4}  {'-'*12} {'-'*40} {'-'*8}  {'-'*10}")

    for i, chat in enumerate(chats, 1):
        restricted = "[LOCKED]" if chat["restricted"] else ""
        name = chat["title"][:37] + "..." if len(chat["title"]) > 40 else chat["title"]
        print(f"  {i:>4}  {chat['type']:<12} {name:<40} {str(chat['members']):>8}  {restricted}")

    print()

## 12. Media Scanning & Size Estimation

### 12.1 Media Count Scanning (Server-Side)

`scan_media_counts()` — Uses `search_messages_count()` per media filter for fast server-side counting. Returns dict of `{label: {count, filter}}` for each media type with >0 files.

In [ ]:
async def scan_media_counts(client, chat_id):
    """Get media breakdown counts using search_messages_count (fast, server-side)."""
    counts = {}
    for label, msg_filter in MEDIA_FILTER_MAP.items():
        try:
            count = await client.search_messages_count(chat_id, filter=msg_filter)
            if count > 0:
                counts[label] = {"count": count, "filter": msg_filter}
        except (FloodWait, FloodPremiumWait) as e:
            await asyncio.sleep(getattr(e, "value", 10))
            try:
                count = await client.search_messages_count(chat_id, filter=msg_filter)
                if count > 0:
                    counts[label] = {"count": count, "filter": msg_filter}
            except (FloodWait, FloodPremiumWait) as e2:
                await asyncio.sleep(getattr(e2, "value", 10))
            except RPCError:
                pass
        except RPCError:
            pass
    return counts

### 12.2 Size Estimation via Sampling

`estimate_type_size()` — Samples up to 50 files per media type using stride-based spread across the channel (not just newest files). Uses average file size × count for total estimate.

In [ ]:
async def estimate_type_size(client, chat_id, msg_filter, count, sample_limit=50):
    """Estimate total size for a media type by sampling files spread across the channel."""
    if count == 0:
        return 0
    # Sample with stride to get files from across the whole channel (not just newest)
    # e.g., 386 files with sample_limit=50 → stride=7, sample every 7th file
    stride = max(1, count // sample_limit)
    sample_sizes = []
    for attempt in range(2):
        try:
            idx = 0
            async for msg in client.search_messages(chat_id, filter=msg_filter):
                if idx % stride == 0:
                    if msg.media:
                        size = get_file_size(msg)
                        if size > 0:
                            sample_sizes.append(size)
                idx += 1
                if len(sample_sizes) >= sample_limit:
                    break
            break  # success
        except (FloodWait, FloodPremiumWait) as e:
            await asyncio.sleep(getattr(e, "value", 10))
        except RPCError:
            break
    if not sample_sizes:
        return 0
    # Use average — correct for estimating totals (median undercounts heavy-tail distributions)
    avg_size = sum(sample_sizes) / len(sample_sizes)
    return int(avg_size * count)

### 12.3 File Listing per Media Type

`list_files_for_type()` — Fetches all messages of a specific media type. Returns list of dicts with message, ID, filename, size, date, and download status. Handles FloodWait with up to 5 retries.

In [ ]:
async def list_files_for_type(client, chat_id, msg_filter, tracker):
    """Fetch all messages of a specific media type with their sizes and names."""
    files = []
    seen_ids = set()

    for attempt in range(5):
        try:
            async for message in client.search_messages(chat_id, filter=msg_filter):
                if not message.media:
                    continue
                if message.id in seen_ids:
                    continue
                seen_ids.add(message.id)

                msg_id = message.id
                already = is_downloaded(tracker, chat_id, msg_id)
                file_name = get_file_name(message)
                file_size = get_file_size(message)

                msg_date = message.date
                date_str = msg_date.strftime("%d-%m-%Y") if msg_date else ""

                files.append({
                    "message": message,
                    "msg_id": msg_id,
                    "file_name": file_name,
                    "file_size": file_size,
                    "date": date_str,
                    "date_sort": msg_date or datetime.min,
                    "already_downloaded": already,
                })
            break  # completed full iteration
        except (FloodWait, FloodPremiumWait) as e:
            wait_time = getattr(e, "value", 10)
            print(f"  !! {type(e).__name__}: waiting {wait_time}s then retrying file list...")
            await asyncio.sleep(wait_time)

    return files

## 13. Download Engine

### 13.1 Single File Download

`download_file()` — Routes to chunked or standard download based on file size (threshold: 50 MB).

Also handles:
- Duplicate filenames (appends `_{msg_id}`, then `_{msg_id}_{counter}`)
- Custom file naming via `FILE_NAME_TEMPLATE`
- Preserves original file dates via `os.utime()`
- Cleans up 0-byte files from failed downloads

In [ ]:
async def download_file(client, file_info, chat_dir, chat_id, chat_title, tracker, download_count):
    """Download a single file with appropriate strategy."""
    message = file_info["message"]
    raw_name = sanitize_name(file_info["file_name"])
    file_name = apply_name_template(FILE_NAME_TEMPLATE, message, raw_name)
    file_size = file_info["file_size"]
    msg_id = file_info["msg_id"]
    file_path = os.path.join(chat_dir, file_name)

    # Handle duplicate filenames (loop in case name_{msg_id} also exists)
    if os.path.exists(file_path):
        name, ext = os.path.splitext(file_name)
        file_name = f"{name}_{msg_id}{ext}"
        file_path = os.path.join(chat_dir, file_name)
        counter = 2
        while os.path.exists(file_path):
            file_name = f"{name}_{msg_id}_{counter}{ext}"
            file_path = os.path.join(chat_dir, file_name)
            counter += 1

    use_chunked = file_size >= LARGE_FILE_THRESHOLD

    if use_chunked:
        print(f"\n  [LARGE {format_size(file_size)}] {file_name}")
        path = await chunked_download(client, message, file_path, file_name, chat_id)
    else:
        path = await standard_download(client, message, file_path, file_name, chat_id)

    if path and os.path.exists(path):
        actual_size = os.path.getsize(path)
        if actual_size == 0:
            # Download produced an empty file — clean it up, don't track
            os.remove(path)
            print(f"  !! {file_name}: downloaded 0 bytes (media unavailable?) — removed")
            return 0
        # Preserve original file date from Telegram message
        if message.date:
            try:
                ts = message.date.timestamp()
                os.utime(path, (ts, ts))
            except Exception:
                pass
        download_count[0] += 1
        mark_downloaded(tracker, chat_id, chat_title, msg_id, download_count[0])
        return actual_size
    else:
        # Download returned None — clean up any leftover empty file
        if os.path.exists(file_path) and os.path.getsize(file_path) == 0:
            os.remove(file_path)
        return 0

### 13.2 Batch Download with Progress Tracking

`download_selected_files()` — Downloads a list of selected files sequentially with:

- **Adaptive delays** between files (0.5s for <1MB, up to 10s for >500MB) to reduce FloodWait
- **Batch progress** with real-time speed and ETA (adaptive frequency based on file size)
- **FloodWait retry** — Sleeps then re-downloads the same file
- **Ctrl+C handler** — Saves tracker, adds remaining files to retry queue
- **Error tracking** — Failed files added to `session_failed_files` for later retry

In [ ]:
async def download_selected_files(client, files, chat_id, chat_title, tracker):
    """Download a list of selected files with delay, speed tracking, and batch progress."""
    chat_dir = os.path.join(DOWNLOAD_DIR, sanitize_name(chat_title))
    os.makedirs(chat_dir, exist_ok=True)

    stats = {"downloaded": 0, "failed": 0, "total_size": 0}
    download_count = [0]
    total_files = len(files)
    total_queued_size = sum(f["file_size"] for f in files)
    batch_start = time.time()
    bytes_done = 0

    for i, file_info in enumerate(files, 1):
        file_name = file_info["file_name"]
        file_size = file_info["file_size"]

        print(f"\n  [{i}/{total_files}] {file_name}")

        try:
            size = await download_file(
                client, file_info, chat_dir, chat_id, chat_title, tracker, download_count
            )
            if size > 0:
                stats["downloaded"] += 1
                stats["total_size"] += size
                bytes_done += size
            else:
                print(f"  !! Media unavailable for msg {file_info['msg_id']}")
                stats["failed"] += 1
                session_failed_files.append({
                    "msg_id": file_info["msg_id"],
                    "file_name": file_name,
                    "file_size": file_size,
                    "chat_id": chat_id,
                    "chat_title": chat_title,
                })

        except FloodWait as e:
            print(f"\n  !! FloodWait: sleeping {e.value}s then retrying...")
            await asyncio.sleep(e.value)
            try:
                size = await download_file(
                    client, file_info, chat_dir, chat_id, chat_title, tracker, download_count
                )
                if size > 0:
                    stats["downloaded"] += 1
                    stats["total_size"] += size
                    bytes_done += size
                else:
                    stats["failed"] += 1
                    session_failed_files.append({
                        "msg_id": file_info["msg_id"],
                        "file_name": file_name,
                        "file_size": file_size,
                        "chat_id": chat_id,
                        "chat_title": chat_title,
                    })
            except Exception as retry_err:
                print(f"\n  !! Retry failed: {retry_err}")
                stats["failed"] += 1
                session_failed_files.append({
                    "msg_id": file_info["msg_id"],
                    "file_name": file_name,
                    "file_size": file_size,
                    "chat_id": chat_id,
                    "chat_title": chat_title,
                })

        except RPCError as e:
            print(f"\n  !! RPC error: {e}")
            stats["failed"] += 1
            session_failed_files.append({
                "msg_id": file_info["msg_id"],
                "file_name": file_name,
                "file_size": file_size,
                "chat_id": chat_id,
                "chat_title": chat_title,
            })

        except KeyboardInterrupt:
            print(f"\n\n  [WARN] Ctrl+C pressed! Saving progress...")
            save_tracker(tracker)
            # Add remaining files (including current) to failed for retry
            for remaining in files[i - 1:]:
                if not is_downloaded(tracker, chat_id, remaining["msg_id"]):
                    session_failed_files.append({
                        "msg_id": remaining["msg_id"],
                        "file_name": remaining["file_name"],
                        "file_size": remaining["file_size"],
                        "chat_id": chat_id,
                        "chat_title": chat_title,
                    })
            remaining_count = len(files) - i + 1
            print(f"  [OK] Saved. {remaining_count} file(s) added to retry queue.")
            return stats

        except Exception as e:
            print(f"\n  !! Failed: {type(e).__name__}: {e}")
            stats["failed"] += 1
            session_failed_files.append({
                "msg_id": file_info["msg_id"],
                "file_name": file_name,
                "file_size": file_size,
                "chat_id": chat_id,
                "chat_title": chat_title,
            })

        # ─── Batch progress (adaptive frequency) ───
        # Small files (<1MB): every 25 files or last file
        # Medium files (<50MB): every 5 files or last file
        # Large files: every file
        show_progress = (i == total_files) or stats["failed"] > 0
        if not show_progress:
            if file_size >= 50 * 1024 * 1024:
                show_progress = True
            elif file_size >= 1 * 1024 * 1024:
                show_progress = (i % 5 == 0)
            else:
                show_progress = (i % 25 == 0)

        if show_progress:
            elapsed = time.time() - batch_start
            real_speed = bytes_done / elapsed if elapsed > 0 else 1
            remaining_bytes = total_queued_size - bytes_done
            eta_seconds = remaining_bytes / real_speed if real_speed > 0 else 0

            print(f"\n  -- Batch: {stats['downloaded']}/{total_files} done | "
                  f"{format_size(bytes_done)}/{format_size(total_queued_size)} | "
                  f"{format_size(real_speed)}/s | "
                  f"ETA: ~{format_time(eta_seconds)}"
                  + (f" | {stats['failed']} failed" if stats['failed'] else "")
                  + " --")

        # Adaptive delay between files based on size (reduces FloodWait)
        if i < total_files:
            if file_size < 1 * 1024 * 1024:
                delay = INTER_DELAY_TINY
            elif file_size < 10 * 1024 * 1024:
                delay = INTER_DELAY_SMALL
            elif file_size < 50 * 1024 * 1024:
                delay = INTER_DELAY_MEDIUM
            elif file_size < 500 * 1024 * 1024:
                delay = INTER_DELAY_LARGE
            else:
                delay = INTER_DELAY_XLARGE
            if delay >= 2:
                print(f"  [pause {delay}s before next file...]")
            await asyncio.sleep(delay)

    save_tracker(tracker)
    return stats

## 14. Channel Processing (Main Download Flow)

`process_chat()` — Orchestrates the full download flow for a single channel:

1. **Scan media counts** → show breakdown with size estimates per type
2. **User selects media types** (numbers, ranges, or all)
3. **Optional Saved Messages scan** to skip already-saved files
4. **Per-type file handling:**
   - Videos/documents → individual file listing with date, size, name (sorted newest first)
   - Photos/audio/voice/gifs → batch yes/no
5. **Download confirmation** → summary with type breakdown, total size, and ETA
6. **Execute downloads** → per-chat summary

In [ ]:
async def process_chat(client, chat_info, tracker):
    """Process a single chat: scan, show breakdown, let user pick what to download."""
    chat_id = chat_info["id"]
    chat_title = chat_info["title"]

    print(f"\n  {'='*60}")
    print(f"  Channel: {chat_title}")
    if chat_info["restricted"]:
        print(f"  [PROTECTED CONTENT -- will download via API]")
    print(f"  {'='*60}")

    # Step 1: Scan media counts
    print(f"\n  Scanning media in this channel...")
    counts = await scan_media_counts(client, chat_id)

    if not counts:
        print(f"  No media found in this channel. Skipping.")
        return {"downloaded": 0, "failed": 0, "total_size": 0}

    # Step 2: Show breakdown with estimated sizes
    total_files = sum(c["count"] for c in counts.values())
    print(f"\n  Media breakdown ({total_files} total files):")
    print(f"  Estimating sizes...")

    type_options = []
    grand_est_size = 0
    for label, info in counts.items():
        est_size = await estimate_type_size(client, chat_id, info["filter"], info["count"])
        info["est_size"] = est_size
        grand_est_size += est_size

    print()
    for i, (label, info) in enumerate(counts.items(), 1):
        mode = "individual" if label in LARGE_MEDIA_TYPES else "batch"
        est = format_size(info.get("est_size", 0))
        print(f"    {i}) {label:<15} {info['count']:>5} files   ~{est:<12} [{mode} selection]")
        type_options.append((label, info))

    print(f"\n    Total estimated size: ~{format_size(grand_est_size)}")

    print(f"\n    A) All media ({total_files} files)")
    print(f"    S) Skip this channel")

    # Step 3: User picks media types
    while True:
        choice = input(f"\n  Select media types (e.g., 1,2 or 1-3 or A for all, S to skip): ").strip()

        if not choice:
            continue
        if choice.upper() == "S":
            print(f"  Skipping {chat_title}.")
            return {"downloaded": 0, "failed": 0, "total_size": 0}
        if choice.upper() == "A":
            selected_types = list(range(len(type_options)))
            break

        selected_types = parse_selection(choice, len(type_options))
        if selected_types is not None:
            break
        print(f"  [ERROR] Invalid input. Use 1-{len(type_options)}, ranges, A, or S.")

    # Step 3.5: Optionally scan Saved Messages to skip already-saved files
    saved_files = set()
    scan_choice = input("\n  Scan Saved Messages to skip already-saved files? (y/n, default=n): ").strip().lower()
    if scan_choice == "y":
        print(f"  Scanning Saved Messages for '{chat_title}'...")
        saved_files = await scan_saved_for_channel(client, chat_title)
        if saved_files:
            print(f"  Found {len(saved_files)} file(s) already in Saved Messages.")
        else:
            print(f"  No files found in Saved Messages for this channel.")
        print(f"  Waiting 5s (rate limit cooldown)...")
        await asyncio.sleep(5)

    # Step 4: For each selected type, handle batch vs individual
    all_selected_files = []
    grand_stats = {"downloaded": 0, "failed": 0, "total_size": 0}

    for type_idx in selected_types:
        label, info = type_options[type_idx]
        msg_filter = info["filter"]
        is_large_type = label in LARGE_MEDIA_TYPES

        print(f"\n  --- {label.upper()} ({info['count']} files) ---")
        print(f"  Fetching file list...")

        try:
            files = await list_files_for_type(client, chat_id, msg_filter, tracker)
        except ChannelPrivate:
            print(f"  !! Removed from channel. Skipping.")
            return grand_stats
        except ChatAdminRequired:
            print(f"  !! Admin rights required. Skipping.")
            return grand_stats
        except (FloodWait, FloodPremiumWait) as e:
            wait_time = getattr(e, "value", 10)
            print(f"  !! {type(e).__name__}: sleeping {wait_time}s then retrying...")
            await asyncio.sleep(wait_time)
            try:
                files = await list_files_for_type(client, chat_id, msg_filter, tracker)
            except Exception as e2:
                print(f"  !! Failed to fetch {label}: {e2}. Skipping type.")
                continue

        new_files = [f for f in files if not f["already_downloaded"]]
        done_count = len(files) - len(new_files)

        # Also filter out files already in Saved Messages (if scan was done)
        if saved_files:
            before = len(new_files)
            new_files = [f for f in new_files if f["file_name"] not in saved_files]
            saved_skip = before - len(new_files)
        else:
            saved_skip = 0

        if done_count > 0 or saved_skip > 0:
            parts = []
            if done_count > 0:
                parts.append(f"{done_count} already downloaded")
            if saved_skip > 0:
                parts.append(f"{saved_skip} already in Saved Messages")
            print(f"  ({' | '.join(parts)} -- will be skipped)")

        if not new_files:
            print(f"  All {label} already downloaded or saved!")
            continue

        known_files = [f for f in new_files if f["file_size"] > 0]
        unknown_count = len(new_files) - len(known_files)
        total_size = sum(f["file_size"] for f in known_files)
        # If some files have unknown sizes, use the estimate from scan as the real total
        if unknown_count > 0 and known_files:
            avg_known = total_size / len(known_files)
            total_size_with_est = total_size + int(avg_known * unknown_count)
        else:
            total_size_with_est = total_size
        est_time = total_size_with_est / ESTIMATED_SPEED if total_size_with_est > 0 else 0

        if is_large_type:
            # ─── INDIVIDUAL SELECTION for videos/documents ───
            # Sort by date (newest first) for easier selection
            new_files.sort(key=lambda f: f.get("date_sort", datetime.min), reverse=True)

            size_note = f"{format_size(total_size_with_est)}"
            if unknown_count > 0:
                size_note += f" (est. — {unknown_count} files have unknown sizes)"
            print(f"\n  {len(new_files)} {label} available ({size_note}, ~{format_time(est_time)}):\n")
            print(f"  {'#':>4}  {'Date':>10}  {'Size':>10}  {'Filename'}")
            print(f"  {'-'*4}  {'-'*10}  {'-'*10}  {'-'*50}")

            for i, f in enumerate(new_files, 1):
                name = f["file_name"]
                if len(name) > 50:
                    name = name[:47] + "..."
                size_str = format_size(f["file_size"]) if f["file_size"] > 0 else "? size"
                date_str = f.get("date", "")
                print(f"  {i:>4}  {date_str:>10}  {size_str:>10}  {name}")

            print(f"\n  Total: {format_size(total_size_with_est)} | Estimated: ~{format_time(est_time)}")
            print()

            while True:
                pick = input(
                    f"  Select {label} (e.g., 1 or 1-3 or 1,2 or all or skip): "
                ).strip().lower()

                if pick == "skip":
                    print(f"  Skipping {label}.")
                    break
                if pick == "all":
                    all_selected_files.extend(new_files)
                    print(f"  Queued all {len(new_files)} {label} ({format_size(total_size)})")
                    break

                indices = parse_selection(pick, len(new_files))
                if indices is not None:
                    selected = [new_files[i] for i in indices]
                    sel_size = sum(f["file_size"] for f in selected)
                    sel_time = sel_size / ESTIMATED_SPEED if sel_size > 0 else 0
                    all_selected_files.extend(selected)
                    print(f"  Queued {len(selected)} {label} ({format_size(sel_size)}, ~{format_time(sel_time)})")
                    break

                print(f"  [ERROR] Invalid. Use 1-{len(new_files)}, ranges, 'all', or 'skip'.")

        else:  # not is_large_type
            # ─── BATCH SELECTION for photos/audio/voice/gifs ───
            size_label = format_size(total_size_with_est)
            if unknown_count > 0:
                size_label += f" (est. — {unknown_count} unknown sizes)"
            print(f"\n  {len(new_files)} {label} ready ({size_label}, ~{format_time(est_time)})")
            print()

            while True:
                pick = input(
                    f"  Download all {len(new_files)} {label}? (y/n/skip): "
                ).strip().lower()

                if pick == "skip" or pick == "n":
                    print(f"  Skipping {label}.")
                    break
                if pick == "y":
                    all_selected_files.extend(new_files)
                    print(f"  Queued all {len(new_files)} {label} ({format_size(total_size)})")
                    break

                print(f"  [ERROR] Enter y, n, or skip.")

    # Step 5: Confirm and download
    if not all_selected_files:
        print(f"\n  No files selected for {chat_title}.")
        return grand_stats

    known = [f for f in all_selected_files if f["file_size"] > 0]
    unknown_total = len(all_selected_files) - len(known)
    total_size = sum(f["file_size"] for f in known)
    if unknown_total > 0 and known:
        total_size += int((total_size / len(known)) * unknown_total)
    est_time = total_size / ESTIMATED_SPEED if total_size > 0 else 0

    # Count by type for summary
    type_counts = {}
    for f in all_selected_files:
        media_type = f["message"].media
        tname = media_type.name.lower() if media_type else "file"
        type_counts[tname] = type_counts.get(tname, 0) + 1

    print(f"\n  {'='*60}")
    print(f"  DOWNLOAD QUEUE for: {chat_title}")
    print(f"  {'='*60}")
    for tname, cnt in type_counts.items():
        print(f"    {tname:<15} {cnt:>5} files")
    print(f"    {'---':>15}")
    print(f"    {'total':<15} {len(all_selected_files):>5} files")
    print(f"    Size:           {format_size(total_size)}")
    print(f"    Estimated time: ~{format_time(est_time)}")
    print(f"  {'='*60}")

    confirm = input(f"\n  Start downloading? (y/n): ").strip().lower()
    if confirm != "y":
        print(f"  Download cancelled for {chat_title}.")
        return grand_stats

    # Download!
    stats = await download_selected_files(client, all_selected_files, chat_id, chat_title, tracker)
    grand_stats["downloaded"] += stats["downloaded"]
    grand_stats["failed"] += stats["failed"]
    grand_stats["total_size"] += stats["total_size"]

    # Per-chat summary
    print(f"\n  -- Summary for: {chat_title}")
    print(f"     Downloaded: {stats['downloaded']} files ({format_size(stats['total_size'])})")
    print(f"     Failed: {stats['failed']}")

    return grand_stats

## 15. Upload Engine (Send to Saved Messages)

### 15.1 Single File Upload

`_upload_single_file()` — Uploads a single file to Saved Messages:

- Auto-selects send method by extension (photo/video/audio/document)
- Upload progress callback for large files (>50 MB)
- Capped FloodWait retries (`UPLOAD_FLOOD_RETRY_CAP=10`) — flood retries don't consume the 3-attempt limit
- Caption set to `ChannelName/filename` for later recovery scanning

In [ ]:
async def _upload_single_file(client, fpath, fname, ch_name, send_as_doc, sent_tracker,
                              stats, stats_lock, batch_bytes, batch_start):
    """Upload a single file to Saved Messages. Used by both sequential and parallel paths."""
    fsize = os.path.getsize(fpath)
    ext = os.path.splitext(fname)[1].lower()
    caption = f"{ch_name}/{fname}"
    if len(caption) > 200:
        caption = caption[:200]

    display = fname if len(fname) <= 40 else fname[:37] + "..."
    is_large = fsize > 50 * 1024 * 1024

    if is_large:
        print(f"\n  Sending {display} ({format_size(fsize)})...")
        upload_start = time.time()
    else:
        print(f"  Sending {display} ({format_size(fsize)})...", end=" ", flush=True)

    def make_progress(f_name):
        ustart = time.time()
        def callback(current, total):
            print_progress(f_name, current, total, ustart)
        return callback

    progress_cb = make_progress(display) if is_large else None
    flood_retries = 0

    for attempt in range(3):
        try:
            if send_as_doc or ext not in (PHOTO_EXTENSIONS | VIDEO_EXTENSIONS | AUDIO_EXTENSIONS):
                await client.send_document("me", fpath, caption=caption, progress=progress_cb)
            elif ext in PHOTO_EXTENSIONS:
                await client.send_photo("me", fpath, caption=caption, progress=progress_cb)
            elif ext in VIDEO_EXTENSIONS:
                await client.send_video("me", fpath, caption=caption, progress=progress_cb)
            elif ext in AUDIO_EXTENSIONS:
                await client.send_audio("me", fpath, caption=caption, progress=progress_cb)

            mark_as_sent(sent_tracker, ch_name, fname)
            if is_large:
                elapsed = time.time() - upload_start
                print(f"\n  [OK] Sent in {format_time(elapsed)}")
            else:
                print("[OK]")
            async with stats_lock:
                stats["sent"] += 1
                batch_bytes[0] += fsize
                # Periodic save
                if stats["sent"] % UPLOAD_SAVE_EVERY == 0:
                    save_sent_tracker(sent_tracker)
            return True

        except (FloodPremiumWait, FloodWait) as e:
            flood_retries += 1
            wait_time = getattr(e, "value", 10)
            error_name = type(e).__name__
            if flood_retries > UPLOAD_FLOOD_RETRY_CAP:
                print(f"\n    {error_name}: exceeded {UPLOAD_FLOOD_RETRY_CAP} flood retries. Skipping {display}.")
                async with stats_lock:
                    stats["failed"] += 1
                return False
            print(f"\n    {error_name}: sleeping {wait_time}s (throttle #{flood_retries})...")
            await asyncio.sleep(wait_time)
            # Don't count flood retries against the attempt limit
            continue

        except RPCError as e:
            if attempt < 2:
                print(f"\n    Error: {e} -- retrying in 5s...")
                await asyncio.sleep(5)
            else:
                print(f"\n    Failed after 3 attempts: {e}")
                async with stats_lock:
                    stats["failed"] += 1
                return False

        except Exception as e:
            if attempt < 2:
                print(f"\n    Error: {e} -- retrying...")
                await asyncio.sleep(3)
            else:
                print(f"\n    Failed: {e}")
                async with stats_lock:
                    stats["failed"] += 1
                return False

    return False

### 15.2 Batch Photo Upload

`_upload_batch_photos()` — Groups up to 10 small photos (<10 MB each) into a single `send_media_group()` API call.

Falls back to individual `_upload_single_file()` sends on batch failure.

In [ ]:
async def _upload_batch_photos(client, batch, ch_name, send_as_doc, sent_tracker,
                               stats, stats_lock, batch_bytes):
    """Upload a batch of photos using send_media_group (up to 10 at once)."""
    if send_as_doc:
        media = [InputMediaDocument(fpath, caption=f"{ch_name}/{fname}")
                 for fname, fpath, fsize in batch]
    else:
        media = [InputMediaPhoto(fpath, caption=f"{ch_name}/{fname}")
                 for fname, fpath, fsize in batch]

    names = [fname for fname, _, _ in batch]
    total_batch_size = sum(fsize for _, _, fsize in batch)
    print(f"  Batch sending {len(batch)} photos ({format_size(total_batch_size)})...", end=" ", flush=True)

    flood_retries = 0
    for attempt in range(3):
        try:
            await client.send_media_group("me", media)
            print(f"[OK]")
            async with stats_lock:
                stats["sent"] += len(batch)
                batch_bytes[0] += total_batch_size
                for fname in names:
                    mark_as_sent(sent_tracker, ch_name, fname)
                if stats["sent"] % UPLOAD_SAVE_EVERY == 0:
                    save_sent_tracker(sent_tracker)
            return True

        except (FloodPremiumWait, FloodWait) as e:
            flood_retries += 1
            wait_time = getattr(e, "value", 10)
            if flood_retries > UPLOAD_FLOOD_RETRY_CAP:
                print(f"\n    Flood retry cap hit. Skipping batch of {len(batch)}.")
                async with stats_lock:
                    stats["failed"] += len(batch)
                return False
            print(f"\n    {type(e).__name__}: sleeping {wait_time}s (throttle #{flood_retries})...")
            await asyncio.sleep(wait_time)
            continue

        except RPCError as e:
            if attempt < 2:
                print(f"\n    Error: {e} -- retrying...")
                await asyncio.sleep(5)
            else:
                print(f"\n    Batch failed: {e}. Falling back to individual sends.")
                # Fallback: send individually
                for fname, fpath, fsize in batch:
                    ok = await _upload_single_file(
                        client, fpath, fname, ch_name, send_as_doc, sent_tracker,
                        stats, stats_lock, batch_bytes, time.time())
                return True  # individual results already tracked

        except Exception as e:
            if attempt < 2:
                await asyncio.sleep(3)
            else:
                print(f"\n    Batch failed: {e}")
                async with stats_lock:
                    stats["failed"] += len(batch)
                return False

    return False

### 15.3 Send to Saved Messages (Main Upload Flow)

`send_to_saved_messages()` — Full upload orchestration:

1. **List downloaded channel folders** with file counts and sizes
2. **Channel selection** (numbers, ranges, or all)
3. **Optional Saved Messages scan** to recover tracker state
4. **Upload mode selection** — Documents (fast, no processing) or Media (with previews)
5. **Per-channel smart menu:** `[1] Send remaining` · `[2] Send all` · `[3] Select` · `[4] Skip`
6. **Filter** — Skips 0-byte files and files exceeding upload limit (2 GB free / 4 GB premium)
7. **Parallel uploads** — `asyncio.Semaphore(3)` with photo batching
8. **Adaptive delays** — No delay for small files; 1s/10s/20s for larger files
9. **Batch progress** with speed and ETA
10. **Ctrl+C handler** — Saves sent tracker on interrupt

In [ ]:
async def send_to_saved_messages(client):
    """Send downloaded files to Telegram Saved Messages with channel-wise selection.

    Features:
    - Parallel uploads (up to UPLOAD_CONCURRENT simultaneous files)
    - Photo batching via send_media_group (up to 10 per API call)
    - Capped FloodWait retries (UPLOAD_FLOOD_RETRY_CAP per file)
    - Reduced inter-file delays (adaptive by size, no delay for small files)
    - Option to send all as documents (faster, skips media processing)
    - Batch progress with speed and ETA
    """
    if not os.path.exists(DOWNLOAD_DIR):
        print("  No downloads directory found.")
        return

    sent_tracker = load_sent_tracker()

    # List downloaded channel folders
    channels = []
    for name in sorted(os.listdir(DOWNLOAD_DIR)):
        path = os.path.join(DOWNLOAD_DIR, name)
        if os.path.isdir(path):
            files = [f for f in os.listdir(path)
                     if os.path.isfile(os.path.join(path, f)) and not f.endswith(".part")]
            if files:
                total_size = sum(os.path.getsize(os.path.join(path, f)) for f in files)
                channels.append({"name": name, "path": path, "files": files, "total_size": total_size})

    if not channels:
        print("  No downloaded files found.")
        return

    print(f"\n  Downloaded channels ({len(channels)}):\n")
    print(f"  {'#':>4}  {'Files':>6}  {'Size':>10}  {'Channel Name'}")
    print(f"  {'-'*4}  {'-'*6}  {'-'*10}  {'-'*40}")

    for i, ch in enumerate(channels, 1):
        print(f"  {i:>4}  {len(ch['files']):>6}  {format_size(ch['total_size']):>10}  {ch['name']}")

    print()

    # Select channels
    while True:
        sel = input("  Select channels to send (e.g., 1,2 or all or back): ").strip()
        if sel.lower() == "back":
            return
        if sel.lower() == "all":
            selected_channels = list(range(len(channels)))
            break
        selected_channels = parse_selection(sel, len(channels))
        if selected_channels is not None:
            break
        print(f"  [ERROR] Invalid input.")

    # Optional: scan Saved Messages to recover tracker state (uses API quota)
    scan_choice = input("  Scan Saved Messages to detect already-sent files? (y/n, default=n): ").strip().lower()
    if scan_choice == "y":
        selected_channel_names = [channels[i]["name"] for i in selected_channels]
        await rebuild_sent_tracker_for_channels(client, sent_tracker, selected_channel_names)
        print("  Waiting 10s before uploads (rate limit cooldown)...")
        await asyncio.sleep(10)

    # Upload mode selection
    print()
    print("  Upload mode:")
    print("    [1] Documents (faster — no media processing)")
    print("    [2] Media (with previews — slower for videos)")
    mode_pick = input("  Choose (1/2, default=1): ").strip()
    send_as_doc = mode_pick != "2"
    if send_as_doc:
        print("  Mode: Documents (fast)")
    else:
        print("  Mode: Media (with previews)")
    print()

    total_sent = 0
    total_fail = 0

    for ch_idx in selected_channels:
        ch = channels[ch_idx]

        # Calculate remaining (unsent) files
        already_sent = set(sent_tracker.get(ch["name"], {}).get("files", []))
        remaining_files = [f for f in ch["files"]
                          if f not in already_sent
                          and os.path.getsize(os.path.join(ch["path"], f)) > 0]
        total_count = len(ch["files"])
        remaining_count = len(remaining_files)

        print(f"\n  --- {ch['name']} ({total_count} files, {format_size(ch['total_size'])}) ---")
        if already_sent:
            print(f"  Already sent: {len(already_sent)}, Remaining: {remaining_count}")

        if remaining_count == 0:
            print(f"  All files already sent. Skipping.")
            continue

        files_to_send = None
        while True:
            if already_sent:
                print(f"  Options: [1] Send remaining {remaining_count}  [2] Send all {total_count}  [3] Select  [4] Skip")
                pick = input(f"  Choose (1/2/3/4): ").strip()
            else:
                print(f"  Options: [1] Send all {total_count}  [2] Select  [3] Skip")
                pick = input(f"  Choose (1/2/3): ").strip()
            if already_sent:
                if pick == "1":
                    files_to_send = remaining_files
                    break
                if pick == "2":
                    files_to_send = ch["files"]
                    break
                if pick == "3":
                    pass
                elif pick == "4":
                    break
                else:
                    print(f"  [ERROR] Enter 1, 2, 3, or 4.")
                    continue
            else:
                if pick == "1":
                    files_to_send = ch["files"]
                    break
                if pick == "2":
                    pass
                elif pick == "3":
                    break
                else:
                    print(f"  [ERROR] Enter 1, 2, or 3.")
                    continue
            # Select mode
            for j, fname in enumerate(ch["files"], 1):
                fsize = os.path.getsize(os.path.join(ch["path"], fname))
                sent_mark = " [SENT]" if fname in already_sent else ""
                name_display = fname if len(fname) <= 45 else fname[:42] + "..."
                print(f"    {j:>4}  {format_size(fsize):>10}  {name_display}{sent_mark}")
            print()
            sel2 = input(f"  Select files (e.g., 1,2,3 or 1-5 or all): ").strip()
            if sel2.lower() == "all":
                files_to_send = ch["files"]
            else:
                indices = parse_selection(sel2, len(ch["files"]))
                if indices:
                    files_to_send = [ch["files"][i] for i in indices]
                else:
                    print("  Invalid selection. Skipping channel.")
            break

        if not files_to_send:
            continue

        # Filter to uploadable files
        upload_limit = UPLOAD_LIMIT_PREMIUM if user_is_premium else UPLOAD_LIMIT_FREE
        limit_str = "4GB" if user_is_premium else "2GB"
        uploadable = []
        for fname in files_to_send:
            if is_already_sent(sent_tracker, ch["name"], fname):
                continue
            fpath = os.path.join(ch["path"], fname)
            fsize = os.path.getsize(fpath)
            if fsize == 0:
                print(f"  SKIP {fname} -- 0 bytes")
                continue
            if fsize > upload_limit:
                print(f"  SKIP {fname} -- exceeds {limit_str}")
                continue
            uploadable.append((fname, fpath, fsize))

        if not uploadable:
            print(f"  No files to upload for {ch['name']}.")
            continue

        total_upload_size = sum(fsize for _, _, fsize in uploadable)
        print(f"\n  Uploading {len(uploadable)} files ({format_size(total_upload_size)}) to Saved Messages...")
        if send_as_doc:
            print(f"  Mode: Documents | Parallel: up to {UPLOAD_CONCURRENT} | Batch photos: up to {UPLOAD_BATCH_SIZE}")
        print()

        stats = {"sent": 0, "failed": 0}
        stats_lock = asyncio.Lock()
        batch_bytes = [0]  # mutable for closure
        batch_start = time.time()
        semaphore = asyncio.Semaphore(UPLOAD_CONCURRENT)

        # Group files: batch consecutive small photos, send large files individually
        tasks = []
        photo_batch = []

        async def flush_photo_batch(batch_to_flush):
            """Send accumulated photo batch."""
            if not batch_to_flush:
                return
            async with semaphore:
                await _upload_batch_photos(
                    client, batch_to_flush, ch["name"], send_as_doc, sent_tracker,
                    stats, stats_lock, batch_bytes)

        async def upload_single(fname, fpath, fsize):
            """Upload a single non-batchable file."""
            async with semaphore:
                await _upload_single_file(
                    client, fpath, fname, ch["name"], send_as_doc, sent_tracker,
                    stats, stats_lock, batch_bytes, batch_start)
                # Adaptive delay for large files only (small files need no delay)
                if fsize > 500 * 1024 * 1024:
                    await asyncio.sleep(SEND_DELAY_LARGE)
                elif fsize > 100 * 1024 * 1024:
                    await asyncio.sleep(SEND_DELAY_MEDIUM)
                elif fsize > 50 * 1024 * 1024:
                    await asyncio.sleep(1)

        try:
            for fname, fpath, fsize in uploadable:
                ext = os.path.splitext(fname)[1].lower()
                is_batchable_photo = (ext in PHOTO_EXTENSIONS and fsize < 10 * 1024 * 1024)

                if is_batchable_photo:
                    photo_batch.append((fname, fpath, fsize))
                    if len(photo_batch) >= UPLOAD_BATCH_SIZE:
                        batch_copy = list(photo_batch)
                        photo_batch.clear()
                        tasks.append(asyncio.ensure_future(flush_photo_batch(batch_copy)))
                else:
                    # Flush any pending photo batch first
                    if photo_batch:
                        batch_copy = list(photo_batch)
                        photo_batch.clear()
                        tasks.append(asyncio.ensure_future(flush_photo_batch(batch_copy)))
                    tasks.append(asyncio.ensure_future(upload_single(fname, fpath, fsize)))

                # Periodically show batch progress and await completed tasks
                if len(tasks) >= UPLOAD_CONCURRENT * 2:
                    done, pending_tasks = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
                    tasks = list(pending_tasks)
                    # Show batch progress
                    elapsed = time.time() - batch_start
                    speed = batch_bytes[0] / elapsed if elapsed > 0 else 0
                    remaining_bytes = total_upload_size - batch_bytes[0]
                    eta = remaining_bytes / speed if speed > 0 else 0
                    total_done = stats["sent"] + stats["failed"]
                    print(f"\n  -- Upload: {stats['sent']}/{len(uploadable)} sent | "
                          f"{format_size(batch_bytes[0])}/{format_size(total_upload_size)} | "
                          f"{format_size(speed)}/s | ETA: ~{format_time(eta)}"
                          + (f" | {stats['failed']} failed" if stats['failed'] else "")
                          + " --")

            # Flush remaining photo batch
            if photo_batch:
                tasks.append(asyncio.ensure_future(flush_photo_batch(photo_batch)))
                photo_batch.clear()

            # Wait for all remaining tasks
            if tasks:
                await asyncio.gather(*tasks)

        except KeyboardInterrupt:
            print(f"\n\n  [Ctrl+C] Upload interrupted. Saving tracker...")
            save_sent_tracker(sent_tracker)
            print(f"  Sent {stats['sent']} files before interrupt.")
            return

        save_sent_tracker(sent_tracker)

        # Final batch stats
        elapsed = time.time() - batch_start
        speed = batch_bytes[0] / elapsed if elapsed > 0 else 0
        print(f"\n  {ch['name']}: Sent {stats['sent']}, Failed {stats['failed']}"
              f" | {format_size(batch_bytes[0])} in {format_time(elapsed)} ({format_size(speed)}/s)")
        total_sent += stats["sent"]
        total_fail += stats["failed"]

    save_sent_tracker(sent_tracker)
    print(f"\n  [OK] Total sent to Saved Messages: {total_sent}, Failed: {total_fail}")

## 16. Retry Failed Downloads

`retry_failed_downloads()` — Retries all failed downloads from the current session:

1. Lists failed files with sizes and source channels
2. Re-fetches messages via `get_messages()` to get fresh file references
3. Downloads using `download_selected_files()` (new failures are re-added to retry list)

In [ ]:
async def retry_failed_downloads(client, tracker):
    """Retry all failed downloads from the current session."""
    global session_failed_files

    if not session_failed_files:
        print("  No failed downloads in this session.")
        return {"downloaded": 0, "failed": 0, "total_size": 0}

    print(f"\n  {len(session_failed_files)} failed file(s) to retry:\n")
    for i, f in enumerate(session_failed_files, 1):
        print(f"    {i}) {f['file_name']} ({format_size(f['file_size'])}) from {f['chat_title']}")

    print()
    confirm = input("  Retry all? (y/n): ").strip().lower()
    if confirm != "y":
        return {"downloaded": 0, "failed": 0, "total_size": 0}

    # Take current failures and clear the list (new failures during retry will be re-added)
    to_retry = list(session_failed_files)
    session_failed_files.clear()

    # Group by chat
    by_chat = {}
    for f in to_retry:
        key = f["chat_id"]
        if key not in by_chat:
            by_chat[key] = {"title": f["chat_title"], "files": []}
        by_chat[key]["files"].append(f)

    total_stats = {"downloaded": 0, "failed": 0, "total_size": 0}

    for chat_id, info in by_chat.items():
        print(f"\n  Retrying {len(info['files'])} file(s) from {info['title']}...")

        # Re-fetch messages to get fresh file references
        refreshed_files = []
        for f in info["files"]:
            try:
                msg = await client.get_messages(chat_id, f["msg_id"])
                if msg and msg.media:
                    refreshed_files.append({
                        "message": msg,
                        "msg_id": f["msg_id"],
                        "file_name": f["file_name"],
                        "file_size": f["file_size"],
                        "already_downloaded": False,
                    })
                else:
                    print(f"  !! Message {f['msg_id']} no longer available. Skipping.")
            except (FloodWait, FloodPremiumWait) as e:
                await asyncio.sleep(getattr(e, "value", 10))
                try:
                    msg = await client.get_messages(chat_id, f["msg_id"])
                    if msg and msg.media:
                        refreshed_files.append({
                            "message": msg,
                            "msg_id": f["msg_id"],
                            "file_name": f["file_name"],
                            "file_size": f["file_size"],
                            "already_downloaded": False,
                        })
                except Exception:
                    print(f"  !! Message {f['msg_id']} still unavailable after flood wait.")
            except RPCError as e:
                print(f"  !! Failed to fetch message {f['msg_id']}: {e}. Retrying in 5s...")
                await asyncio.sleep(5)
                try:
                    msg = await client.get_messages(chat_id, f["msg_id"])
                    if msg and msg.media:
                        refreshed_files.append({
                            "message": msg,
                            "msg_id": f["msg_id"],
                            "file_name": f["file_name"],
                            "file_size": f["file_size"],
                            "already_downloaded": False,
                        })
                    else:
                        print(f"  !! Message {f['msg_id']} unavailable after retry.")
                except Exception:
                    print(f"  !! Message {f['msg_id']} still failing. Skipping.")

        if refreshed_files:
            stats = await download_selected_files(client, refreshed_files, chat_id, info["title"], tracker)
            total_stats["downloaded"] += stats["downloaded"]
            total_stats["failed"] += stats["failed"]
            total_stats["total_size"] += stats["total_size"]
        else:
            print(f"  !! Could not refresh any messages for {info['title']}. All {len(info['files'])} file(s) skipped.")

    print(f"\n  Retry summary: Downloaded {total_stats['downloaded']}, "
          f"Still failed {total_stats['failed']}")
    return total_stats

## 17. Channel Stats Dashboard

`show_channel_stats()` — Displays per-channel download statistics:
- File counts per channel (sorted by count, descending)
- Last updated timestamps
- Total disk usage across all downloaded files

In [ ]:
def show_channel_stats(tracker):
    """Display per-channel download statistics dashboard."""
    if not tracker:
        print("  No download history yet.")
        return

    print(f"\n  {'='*70}")
    print(f"  CHANNEL STATS DASHBOARD")
    print(f"  {'='*70}")
    print(f"  {'#':>4}  {'Files':>7}  {'Last Updated':<22}  {'Channel Name'}")
    print(f"  {'-'*4}  {'-'*7}  {'-'*22}  {'-'*35}")

    total_files = 0
    for i, (chat_id, info) in enumerate(sorted(tracker.items(), key=lambda x: len(x[1].get("downloaded_ids", [])), reverse=True), 1):
        title = info.get("title", "Unknown")
        count = len(info.get("downloaded_ids", []))
        updated = info.get("last_updated", "N/A")
        if updated and updated != "N/A":
            try:
                dt = datetime.fromisoformat(updated)
                updated = dt.strftime("%Y-%m-%d %H:%M")
            except (ValueError, TypeError):
                pass
        total_files += count
        name = title[:32] + "..." if len(title) > 35 else title
        print(f"  {i:>4}  {count:>7}  {updated:<22}  {name}")

    print(f"  {'-'*70}")
    print(f"  Total: {total_files} files across {len(tracker)} channels")

    # Disk usage
    if os.path.exists(DOWNLOAD_DIR):
        total_disk = 0
        for dirpath, dirnames, filenames in os.walk(DOWNLOAD_DIR):
            for f in filenames:
                if not f.endswith(".part"):
                    total_disk += os.path.getsize(os.path.join(dirpath, f))
        print(f"  Disk usage: {format_size(total_disk)}")
    print(f"  {'='*70}")

## 18. Main Menu & Application Loop

`main()` — Persistent menu loop with 7 options:

| # | Option | Description |
|---|--------|-------------|
| 1 | Select channels to download | Browse channels, pick media types, select files, download |
| 2 | Refresh channel list | Re-fetch from Telegram (if you joined new channels) |
| 3 | Show session stats | Files downloaded, failed, size, elapsed time |
| 4 | Send files to Saved Messages | Upload downloaded files to Telegram Saved Messages |
| 5 | Retry failed downloads | Retry with fresh file references |
| 6 | Channel stats dashboard | Per-channel file counts, disk usage |
| 7 | Exit | Save tracker and disconnect |

In [ ]:
async def main():
    """Main entry point."""
    print()
    print("  +======================================================+")
    print("  |         Telegram Media Downloader v2                 |")
    print("  |     Download media from your channels/groups         |")
    print("  |     (including restricted/protected content)         |")
    print("  +======================================================+")
    if IS_COLAB:
        print(f"  | ENV: Google Colab (high-speed download)              |")
        print(f"  | SAVE TO: /content/telegram_downloader/               |")
        print(f"  +======================================================+")
    print()

    client = await setup_client()
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    tracker = load_tracker()
    session_total = {"downloaded": 0, "failed": 0, "total_size": 0, "chats": 0}
    session_start = time.time()

    try:
        # Fetch chats once
        chats = await list_chats(client)
        if not chats:
            print("  [ERROR] No channels or groups found. Exiting.")
            return

        # ─── Main Menu Loop ──────────────────────────────────────
        while True:
            fail_count = len(session_failed_files)
            print(f"\n  {'='*60}")
            print(f"  MAIN MENU")
            print(f"  {'='*60}")
            print(f"    1) Select channels to download")
            print(f"    2) Refresh channel list")
            print(f"    3) Show session stats")
            print(f"    4) Send files to Saved Messages")
            print(f"    5) Retry failed downloads" + (f" ({fail_count} failed)" if fail_count else ""))
            print(f"    6) Channel stats dashboard")
            print(f"    7) Exit")
            print(f"  {'='*60}")

            menu = input("\n  Choose option (1-7): ").strip()

            if menu == "7":
                break

            elif menu == "6":
                show_channel_stats(tracker)
                continue

            elif menu == "3":
                elapsed = time.time() - session_start
                print(f"\n  Session stats:")
                print(f"     Chats processed:  {session_total['chats']}")
                print(f"     Files downloaded: {session_total['downloaded']}")
                print(f"     Files failed:     {session_total['failed']}")
                print(f"     Total size:       {format_size(session_total['total_size'])}")
                print(f"     Session time:     {format_time(elapsed)}")
                print(f"     Output directory: {DOWNLOAD_DIR}")
                if session_failed_files:
                    print(f"     Pending retries:  {len(session_failed_files)} files")
                continue

            elif menu == "2":
                print()
                chats = await list_chats(client)
                if not chats:
                    print("  [ERROR] No channels or groups found.")
                    continue
                display_chats(chats)
                continue

            elif menu == "4":
                await send_to_saved_messages(client)
                continue

            elif menu == "5":
                retry_stats = await retry_failed_downloads(client, tracker)
                session_total["downloaded"] += retry_stats["downloaded"]
                session_total["failed"] += retry_stats["failed"]
                session_total["total_size"] += retry_stats["total_size"]
                save_tracker(tracker)
                continue

            elif menu == "1":
                display_chats(chats)

                # User selects channels
                while True:
                    selection = input("  Select chats to process (e.g., 1,3,5 or 1-10 or all): ").strip()
                    selected = parse_selection(selection, len(chats))
                    if selected is not None:
                        break
                    print(f"  [ERROR] Invalid input. Use numbers 1-{len(chats)}, ranges, or 'all'.\n")

                selected_chats = [chats[i] for i in selected]

                print(f"\n  Selected {len(selected_chats)} chat(s):")
                for chat in selected_chats:
                    tag = " [PROTECTED]" if chat["restricted"] else ""
                    print(f"    - {chat['title']}{tag}")
                print()

                for i, chat_info in enumerate(selected_chats, 1):
                    print(f"\n  Processing chat {i}/{len(selected_chats)}...")

                    try:
                        stats = await process_chat(client, chat_info, tracker)
                        session_total["downloaded"] += stats["downloaded"]
                        session_total["failed"] += stats["failed"]
                        session_total["total_size"] += stats["total_size"]
                        session_total["chats"] += 1
                    except ChannelPrivate:
                        print(f"  !! Removed from channel. Skipping.")
                    except ChatAdminRequired:
                        print(f"  !! Admin rights required. Skipping.")
                    except RPCError as e:
                        print(f"  !! API error: {e}")

                save_tracker(tracker)

                # Batch completion summary after all selected chats
                batch_time = time.time() - session_start
                print(f"\n  {'='*60}")
                print(f"  DOWNLOAD BATCH COMPLETE")
                print(f"  {'='*60}")
                print(f"     Chats processed:  {len(selected_chats)}")
                print(f"     Session total:    {session_total['downloaded']} downloaded, "
                      f"{session_total['failed']} failed")
                print(f"     Session size:     {format_size(session_total['total_size'])}")
                print(f"     Session time:     {format_time(batch_time)}")
                if session_failed_files:
                    print(f"     Retry available:  {len(session_failed_files)} failed file(s) "
                          f"(option 5)")
                print(f"  {'='*60}")

            else:
                print("  [ERROR] Invalid option. Enter 1-7.")

        # Final summary on exit
        save_tracker(tracker)
        session_time = time.time() - session_start

        print(f"\n  {'='*60}")
        print(f"  SESSION SUMMARY")
        print(f"  {'='*60}")
        print(f"     Chats processed:  {session_total['chats']}")
        print(f"     Files downloaded: {session_total['downloaded']}")
        print(f"     Files failed:     {session_total['failed']}")
        print(f"     Total size:       {format_size(session_total['total_size'])}")
        print(f"     Time taken:       {format_time(session_time)}")
        print(f"     Output directory: {DOWNLOAD_DIR}")
        print(f"  {'='*60}")
        print()

    except KeyboardInterrupt:
        print("\n\n  [WARN] Interrupted by user. Saving progress...")
        try:
            save_tracker(tracker)
            print("  [OK] Progress saved. Re-run to resume.\n")
        except Exception:
            print("  [WARN] Could not save tracker.\n")
    finally:
        global _active_client
        # Disconnect Pyrogram client
        try:
            await client.stop()
            print("  Disconnected from Telegram.\n")
        except Exception:
            print("  Disconnected.\n")
        _active_client = None

## 19. Run the Downloader

Entry point that handles:
- **Windows**: Sets `WindowsSelectorEventLoopPolicy` for asyncio compatibility
- **All platforms**: Launches `main()` via `asyncio.run()`
- **Colab**: Also runs when the cell is executed directly (not imported as a module)

> **Run this cell to start the downloader.**

In [ ]:
def run():
    """Run the downloader, handling both local and Colab environments."""
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    asyncio.run(main())

if __name__ == "__main__":
    run()
else:
    # When pasted directly into a Colab cell (not run as __main__)
    if IS_COLAB:
        run()